# Dimensia AI — Google Colab Android APK Builder (8.3)

**Telefon / Bilgisayar:** Dimensia AI'de **📱 APK Stüdyo → ZIP Projeyi İndir** düğmesini kullan. Colab not defterinde **Çalışma zamanı → Tümünü çalıştır** seç. İlk hücrede indirilen `.zip` dosyasını yükle, ardından kuruluma ve derlemeye devam et.

Bu not defteri Expo / React Native projesini Android SDK + Gradle ile **debug APK** olarak derlemeyi dener. APK başarılıysa son hücre telefona indirir.

**Güvenlik:** Projede gizli anahtar bulunmamalı. Colab hesabını veya tokenlerini Dimensia sitesine girme. Google Colab ücretsiz hesaplarda kaynak ve oturum süresini garanti etmez; Android derlemesi CPU/RAM/kota nedeniyle başarısız olabilir. Bu, Play Store için imzalanmış üretim sürümü değildir.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import io
import shutil

print("📦 Dimensia AI Studio'dan indirdiğin proje ZIP dosyasını yükle.")
incoming = files.upload()
candidates = [(name, data) for name, data in incoming.items() if name.lower().endswith(".zip")]
if len(candidates) != 1:
    raise ValueError("Lütfen sadece bir .zip dosyası yükle.")

name, data = candidates[0]
if len(data) > 2_500_000:
    raise ValueError("Proje ZIP dosyası 2.5 MB sınırını aşıyor.")

project = Path("/content/dimensia-apk-project")
if project.exists():
    shutil.rmtree(project)
project.mkdir(parents=True)
allowed = {"App.js", "App.tsx", "app.json", "package.json", "babel.config.js", "eas.json", "tsconfig.json", "README.md"}
total = 0
with ZipFile(io.BytesIO(data)) as bundle:
    for item in bundle.infolist():
        path = PurePosixPath(item.filename)
        if item.is_dir():
            continue
        if len(path.parts) != 1 or path.name not in allowed or path.is_absolute():
            raise ValueError(f"ZIP içindeki dosya izinli değil: {item.filename}")
        if item.file_size > 750_000:
            raise ValueError(f"Dosya çok büyük: {item.filename}")
        total += item.file_size
        if total > 2_000_000:
            raise ValueError("ZIP toplam kaynak limiti aşıldı.")
        # Reject Unix symbolic links.
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Sembolik bağlantı içeren ZIP kabul edilmez.")
        (project / path.name).write_bytes(bundle.read(item))

for file in ("package.json", "app.json", "babel.config.js"):
    if not (project / file).exists():
        raise ValueError(f"{file} eksik, Dimensia ZIP dosyasını yeniden indir.")
if not (project / "App.js").exists() and not (project / "App.tsx").exists():
    raise ValueError("App.js veya App.tsx bulunamadı.")
print("✅ Proje alındı:", name, "| Dosyalar:", sorted(p.name for p in project.iterdir()))

## 2. Node.js, JDK ve Android SDK kurulumu

Bu adım paketleri Colab oturumuna kurar. İnternet ve Colab kaynaklarına bağlıdır. GPU gerekli değildir. Google'ın Colab çalışma zamanı kapanırsa yeniden çalıştır.

In [ ]:
import os
import subprocess
from pathlib import Path
import urllib.request
import zipfile

def command(argv, cwd=None, env=None, input_text=None):
    print("▶", " ".join(str(x) for x in argv))
    subprocess.run(argv, cwd=cwd, env=env, input=input_text, text=True, check=True)

command(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless curl unzip ca-certificates"])
command(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x -o /tmp/dimensia-nodesource.sh && bash /tmp/dimensia-nodesource.sh && apt-get install -y -qq nodejs"])
command(["node", "--version"])
command(["npm", "--version"])
command(["java", "-version"])

sdk = Path("/content/android-sdk")
tools_dir = sdk / "cmdline-tools" / "latest"
bin_sdkmanager = tools_dir / "bin" / "sdkmanager"
if not bin_sdkmanager.exists():
    sdk.mkdir(parents=True, exist_ok=True)
    archive = Path("/content/android-commandline-tools.zip")
    url = "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip"
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall("/content/android-sdk-temp")
    tools_dir.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.move("/content/android-sdk-temp/cmdline-tools",tools_dir.as_posix())

env = dict(os.environ)
env["ANDROID_HOME"] = sdk.as_posix()
env["ANDROID_SDK_ROOT"] = sdk.as_posix()
env["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
env["PATH"] = str(sdk / "platform-tools") + ":" + env["JAVA_HOME"] + "/bin:" + env["PATH"]
command([str(bin_sdkmanager), "--sdk_root="+str(sdk), "--licenses"], env=env, input_text="y\n"*70)
command([str(bin_sdkmanager), "--sdk_root="+str(sdk), "platform-tools", "platforms;android-36", "build-tools;36.0.0"], env=env)
print("✅ Android SDK hazır.")

## 3. Expo dosyalarından Android debug APK derle

İlk derleme büyük bağımlılıklar indirebilir. Hata varsa son kırmızı çıktıyı Dimensia sohbetine yapıştırıp düzeltme iste. Modifiye edilmemiş projeleri körü körüne çalıştırma; paketler npm komutları yürütebilir.

In [ ]:
from pathlib import Path
import os

project = Path("/content/dimensia-apk-project")
if not project.exists():
    raise ValueError("Önce proje ZIP dosyasını yükle.")
env = dict(os.environ)
env["ANDROID_HOME"] = "/content/android-sdk"
env["ANDROID_SDK_ROOT"] = "/content/android-sdk"
env["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
env["PATH"] = env["JAVA_HOME"] + "/bin:" + env["ANDROID_HOME"] + "/platform-tools:" + env["PATH"]
command(["npm", "install", "--no-audit", "--no-fund"], cwd=project, env=env)
command(["npx", "expo", "prebuild", "--platform", "android", "--non-interactive", "--clean"], cwd=project, env=env)
gradle = project / "android" / "gradlew"
gradle.chmod(gradle.stat().st_mode | 0o111)
command([str(gradle), ":app:assembleDebug", "--no-daemon", "--stacktrace"], cwd=gradle.parent, env=env)
apk = project / "android" / "app" / "build" / "outputs" / "apk" / "debug" / "app-debug.apk"
if not apk.exists():
    raise FileNotFoundError("Gradle tamamlandı ancak debug APK bulunamadı.")
print("✅ APK üretildi:", apk, "|",round(apk.stat().st_size/1024/1024,2),"MB")

In [ ]:
from google.colab import files
from pathlib import Path
apk = Path("/content/dimensia-apk-project/android/app/build/outputs/apk/debug/app-debug.apk")
if not apk.is_file():
    raise FileNotFoundError("APK hazır değil. Önce derleme hücresini çalıştır.")
print("📲 APK indiriliyor. İndirdikten sonra Android'de kurulum izni vermen gerekebilir.")
files.download(apk.as_posix())

## Alternatif: Expo EAS Build

Colab oturumu yetmezse Dimensia ZIP'ini bilgisayarında açıp `npm install`, `npm install -g eas-cli`, `eas login`, `eas build -p android --profile preview` komutlarını kullanabilirsin. `eas.json` içinde `android.buildType=apk` hazırlanmıştır. Expo hesabı ve EAS ücretsiz kotası gerekebilir. Tamamen otomatik veya sınırsız ücretsiz derleme garantisi yoktur.

**Önemli:** Bu notebook kişisel/etkileşimli Colab kullanımı içindir; görünmez arka plan işçisi, Colab API sunucusu veya uzun süreli robot çalışma zamanı başlatmaz.